# PySpark
## Hadoop, Spark
### Hadoop
Apache Hadoop is a distributed system designed to store and process huge datasets across many machines. It contains 3 key components:
- **HDFS (Hadoop Distributed File System)**: it splits big files into blocks and replicates them across machines so data is durable and can be read in parallel.
- **YARN (Yet Another Resource Negotiator)**: it's resource manager and scheduler; it allocates CPU/memory across the cluster and launches jobs/containers. YARN is the “cluster OS,” and Spark is the app. When you run Spark on YARN, Spark talks to YARN through an integration layer, so most YARN details are inferred or handled by the cluster’s defaults.
- **MapReduce (classic Hadoop compute engine)**： parallel computation (batch-oriented) part. This is the original “Hadoop does parallel computation” part. Before Hadoop.2x+, **Hadoop** is synonymous with **MapReduce**.

In [2]:
from collections import defaultdict

def mapper(line: str):
    # Map: emit (word, 1) for each word
    for w in line.strip().split():
        w = ''.join(ch for ch in w.lower() if ch.isalnum())
        if w:
            yield (w, 1)

def reducer(word: str, counts):
    # Reduce: aggregate all counts for the same word
    yield (word, sum(counts))

def mapreduce(lines):
    # 1) MAP
    intermediate = []
    for line in lines:
        intermediate.extend(mapper(line))

    # 2) SHUFFLE/GROUP (group values by key)
    grouped = defaultdict(list)
    for k, v in intermediate:
        grouped[k].append(v)

    # 3) REDUCE
    results = []
    for k in sorted(grouped.keys()):
        results.extend(reducer(k, grouped[k]))
    return results

In [2]:
text = [
        "To be, or not to be: that is the question.",
        "Whether 'tis nobler in the mind to suffer",
        "The slings and arrows of outrageous fortune",
    ]
for word, count in mapreduce(text):
    print(word, count)

and 1
arrows 1
be 2
fortune 1
in 1
is 1
mind 1
nobler 1
not 1
of 1
or 1
outrageous 1
question 1
slings 1
suffer 1
that 1
the 3
tis 1
to 3
whether 1


### Spark
Apache Spark is an open-source unified analytics engine for large-scale data processing (ibraries for SQL, streaming, ML, and graph). In many real steps
- Hadoop provides storage + cluster scheduling
- Spark provides the computation layer

Spark can run without Hadoop at all. You can mostly treat it as a parallel term as MapReduce:
- MapReduce is a distributed computing model/engine (traditionally Hadoop MapReduce) for batch jobs.
- Spark is also a distributed computing engine, but more general (DAG execution, in-memory caching, interactive SQL, streaming, ML libraries). Spark isn’t limited to MapReduce’s two-stage model: it supports general DAGs of transformations, not just map + reduce.

## Run PySpark on EC2
1. Start you EC2 instance, then SSH to your EC2 by
```
ssh -i YOUR_KEY_PATH -L 8888:localhost:8888 EC2_IP_ADDRESS
```

2. Set Docker permissions using the command
```
sudo chmod 666 /var/run/docker.sock 
```

3. Under you course directory on your EC2 instance, create a folder for this lab
```
mkdir {lab5_folder}
cd {lab5_folder}
```

4. In this folder, create file with name Dockerfile that contains the following text:

you can use vim to create such file

```
vim Dockerfile
```

The copy the following Dockerfile content:
```
FROM jupyter/datascience-notebook

WORKDIR /home/jovyan
EXPOSE 8888

ENV DEBIAN_FRONTEND=noninteractive

USER root

RUN apt-get update && apt-get install -y \
    wget \
    curl \
    unzip \
    bzip2 \
    git \
    git-lfs \
    build-essential \
    libopenblas-dev \
    openjdk-17-jdk \
    && rm -rf /var/lib/apt/lists/*

# Use default Java location provided by Debian/Ubuntu
ENV JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
ENV PATH="${JAVA_HOME}/bin:${PATH}"

# Install AWS CLI v2
RUN ARCH=$(uname -m) && \
    if [ "$ARCH" = "x86_64" ]; then \
        AWSCLI_ARCH="x86_64"; \
    elif [ "$ARCH" = "aarch64" ]; then \
        AWSCLI_ARCH="aarch64"; \
    else \
        echo "Unsupported architecture: $ARCH" && exit 1; \
    fi && \
    curl "https://awscli.amazonaws.com/awscli-exe-linux-${AWSCLI_ARCH}.zip" -o "awscliv2.zip" && \
    unzip awscliv2.zip && \
    ./aws/install && \
    rm -rf awscliv2.zip ./aws

RUN pip install --no-cache-dir --upgrade pip && \
    pip install --no-cache-dir \
      torch --index-url https://download.pytorch.org/whl/cpu && \
    pip install --no-cache-dir \
      transformers datasets evaluate accelerate && \
    pip install --no-cache-dir \
      boto3 s3fs fsspec implicit && \
    pip install --no-cache-dir \
      pyspark findspark pyarrow

ENV PYSPARK_PYTHON=python
ENV PYSPARK_DRIVER_PYTHON=python
ENV SPARK_LOCAL_IP=127.0.0.1

COPY . .

RUN chown -R jovyan:users /home/jovyan

USER jovyan

CMD ["start-notebook.sh", "--NotebookApp.token=''", "--NotebookApp.password=''"]
```

To build and run it on EC2:
```
docker build -t jupyter-pyspark-ec2 .
```

Then run the container
```
docker run -p 8888:8888 \
  -v $(pwd):/home/jovyan/work \
  jupyter-pyspark-ec2
```

Note: with this command, all files in your {lab5_folder} will be synchronised with files in `work` folder in Jupyter Lab.

Then open in your browser:
```
http://<EC2_PUBLIC_IP>:8888
```